# 1. Objective
This notebook generates our second synthetic dataset using **CTGAN (Conditional Tabular GAN)**. CTGAN is a deep learning-based generator explicitly designed for mixed-type tabular data. It will be trained strictly on the `adult.data` (training) dataset. We will generate approximately the same number of rows as the original training dataset (32,561 rows) and save it for future evaluation.

# 2. Imports

In [1]:
import pandas as pd
import numpy as np
import torch
import warnings
import sdv
from sdv.metadata import SingleTableMetadata
from sdv.single_table import CTGANSynthesizer
import os

warnings.filterwarnings('ignore')
print(f"SDV version: {sdv.__version__}")

SDV version: 1.38.3


# 3. Load Training Data
We load the exact same 15-column `adult.data` dataset used in our previous experiments. Missing values (`?`) are parsed as `NaN`. `adult.test` is strictly avoided.

In [2]:
columns = [
    "age", "workclass", "fnlwgt", "education", "education-num", 
    "marital-status", "occupation", "relationship", "race", "sex", 
    "capital-gain", "capital-loss", "hours-per-week", "native-country", "income"
]

# Load ONLY the training dataset
df = pd.read_csv('../data/raw/adult.data', names=columns, skipinitialspace=True, na_values='?')
print(f"Loaded {len(df)} records from real training dataset.")
df.head()

Loaded 32561 records from real training dataset.


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


# 4. Prepare SDV Metadata
We use SDV's `SingleTableMetadata` to automatically infer the types of the columns (numerical vs categorical). CTGAN natively handles mixed types internally using Variational Gaussian Mixture Models (VGM) for continuous columns and Gumbel-Softmax for categoricals, so we do not manually one-hot encode or standardize the data.

In [3]:
metadata = SingleTableMetadata()
metadata.detect_from_dataframe(data=df)

# 5. Configure CTGAN
We initialize the `CTGANSynthesizer`. We configure it with a reasonable research baseline of `epochs=50` to balance deep learning convergence with computational time. (The default is 300, which can take hours on CPUs for 32k rows).

In [4]:
synthesizer = CTGANSynthesizer(
    metadata,
    enforce_rounding=True,
    epochs=50,
    verbose=True
)

# Set fixed random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# 6. Train CTGAN
We fit the deep learning model purely on the original training dataset. This may take a few minutes depending on CPU/GPU availability.

In [5]:
synthesizer.fit(df)

Gen. (-00.38) | Discrim. (-00.41): 100%|██████████| 50/50 [05:34<00:00,  6.68s/it]


# 7. Generate Synthetic Data
We sample exactly 32,561 records from the trained Generator network.

In [6]:
synthetic_data = synthesizer.sample(num_rows=len(df))

# 8. Save Synthetic Dataset
We export the dataset to our `data/synthetic` directory.

In [7]:
os.makedirs('../data/synthetic', exist_ok=True)
output_path = '../data/synthetic/adult_ctgan.csv'
synthetic_data.to_csv(output_path, index=False)
print(f"Saved to: {output_path}")

Saved to: ../data/synthetic/adult_ctgan.csv


# 9. Basic Verification
We perform a structural sanity check on the output.

In [8]:
print("--- Real Data Dimensions ---")
print(df.shape)
print("\n--- Synthetic Data Dimensions ---")
print(synthetic_data.shape)

print("\n--- Synthetic Data Missing Values ---")
missing = synthetic_data.isnull().sum()
print(missing[missing > 0])

print("\n--- Synthetic Target Distribution ---")
print(synthetic_data['income'].value_counts(normalize=True))

print("\n--- Basic Numerical Ranges ---")
print(synthetic_data[['age', 'hours-per-week']].describe().loc[['min', 'max']])

print(f"\nFile exists: {os.path.exists(output_path)}")
synthetic_data.head()

--- Real Data Dimensions ---
(32561, 15)

--- Synthetic Data Dimensions ---
(32561, 15)

--- Synthetic Data Missing Values ---
workclass         2564
occupation        2388
native-country     894
dtype: int64

--- Synthetic Target Distribution ---
income
<=50K    0.800835
>50K     0.199165
Name: proportion, dtype: float64

--- Basic Numerical Ranges ---
      age  hours-per-week
min  17.0             3.0
max  90.0            99.0

File exists: True


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,26,Private,358077,HS-grad,9,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,40,United-States,<=50K
1,22,NaN,297768,Some-college,9,Never-married,NaN,Own-child,White,Male,17,1,17,United-States,<=50K
2,31,Private,363423,Some-college,10,Never-married,Adm-clerical,Own-child,White,Male,0,0,12,United-States,<=50K
3,38,Federal-gov,149521,Masters,5,Married-civ-spouse,Prof-specialty,Husband,White,Female,0,0,40,United-States,>50K
4,36,Private,109903,Some-college,10,Never-married,NaN,Other-relative,White,Female,0,0,19,United-States,<=50K


# 10. Experiment Summary
- **Generator**: CTGAN (`CTGANSynthesizer` from SDV v1.38.3).
- **Configuration**: 50 Epochs, enforce_rounding=True, random_seed=42.
- **Rows Generated**: 32,561 (matching the real training dataset exactly).
- **Columns**: 15 (matching the real training dataset exactly).
- **Output Path**: `data/synthetic/adult_ctgan.csv`.
- **Constraint Checks**: `adult.test` was strictly avoided. The original `adult.data` raw files were loaded dynamically and not modified on disk.
- **Status**: The CTGAN baseline is structurally sound and ready for formal fidelity, utility, and privacy evaluation in subsequent notebooks.